In [ ]:
!pip install natsort

In [ ]:
import shutil
from glob import glob
import os
import re
from moviepy.editor import VideoFileClip, concatenate_videoclips
from natsort import natsorted

# паттерн для идентификатора мышесессии
pattern = r'[A-z0-9]{3,4}_[A-z]{1,2}\d{2}_(\dD|\dT)(_\dT)?'

# укажи путь к папке с папками мышейсессий, добавь двойные слеши
root = 'w:\\Projects\\3DM\\Comparasion\\3DMaze\\1_RawBehaviordata\\Sandbox\\'

# выбери структуру папок
# For Bonsai data
timestamps = glob(root + '*//*.csv')
# or For MiniscopeSoft data
#timestamps = glob(root + '*//*//*//Miniscope//timeStamps.csv')

print(f"Найдено всего {len(timestamps)} таймстемпов")
timestamps

In [ ]:
for tm in timestamps:
    splitted_path = os.path.normpath(tm).split(os.sep)

    # Поиск индекса каталога, совпадающего с шаблоном
    identifier_index = None
    for i, directory in enumerate(splitted_path):
        if re.match(pattern, directory):
            print(directory)
            identifier_index = i
            break
    
    # Вывод результата
    if identifier_index is not None:
        print(f"Идентификатор найден: '{splitted_path[identifier_index]}'")
    else:
        print("Идентификатор не найден в пути.")    
    
    mouse_name = splitted_path[identifier_index]
            
    # Считывает файлы и сортирует          
    avi_files = glob(os.path.dirname(tm) + '\\*.avi')    
    avi_files = natsorted(avi_files)
    print("\n".join(avi_files))
    
    # Объединяет видео и сохраняет
    clips = []
    for av in avi_files:            
        clips.append(VideoFileClip(av))
    result = concatenate_videoclips(clips, method='compose')    
    
    # поменять для поведения 
    result.write_videofile(root + mouse_name + '.mp4')


In [ ]:
## import shutil
# ToDo naming for 1 mode is wrong (only mouse id)

from glob import glob
import os
import re
from moviepy.editor import VideoFileClip, concatenate_videoclips
from natsort import natsorted

# Паттерн для идентификатора мышесессии вида "MSS_F01_1D_1T"
pattern = r'[A-Z]{3,4}_[A-Z]\d{2}_(\dD|\dT)(_\dT)?'

# Укажите путь к корневой папке с папками мышесессий
exp = 'HOS'
root = 'w:\\Projects\\HOS\\BehaviorData\\1_Raw\\'

# Выберите режим работы:
# 1 - Сшить все видео из одной папки
#mode = '1'
# 2 - Сшить все видео, относящиеся к одной мыши
mode = '1'

timestamps = glob(root + '*//*.csv')
print(f"Найдено всего {len(timestamps)} таймстемпов")
timestamps

if mode not in ["1", "2"]:
    print("Неверный выбор режима. Завершение программы.")
    exit()

# Множество для отслеживания обработанных идентификаторов мышей (только для режима 2)
processed_mice = set()

for tm in timestamps:
    splitted_path = os.path.normpath(tm).split(os.sep)

    # Поиск идентификатора мышесессии
    identifier_index = None
    for i, directory in enumerate(splitted_path):
        if re.match(pattern, directory):
            identifier_index = i
            break
    
    if identifier_index is None:
        print(f"Идентификатор не найден в пути: {tm}")
        continue
    
    mouse_session = splitted_path[identifier_index]
    
    # Извлекаем идентификатор мыши (например, F04 из MSS_F04_1D_1T)
    mouse_id_match = re.search(r'_([A-Z]\d{2})_', mouse_session)
    if not mouse_id_match:
        print(f"Не удалось извлечь идентификатор мыши из: {mouse_session}")
        continue
    mouse_id = mouse_id_match.group(1)

    # Проверка для режима 2: пропускаем уже обработанных мышей
    if mode == "2" and mouse_id in processed_mice:
        continue

    # Получение списка видео в зависимости от режима
    avi_files = []
    if mode == "1":  # Режим 1: Сшить видео из одной папки
        avi_files = glob(os.path.dirname(tm) + '\\*.avi')
    elif mode == "2":  # Режим 2: Сшить все видео по идентификатору мыши
        avi_files = glob(root + '**/*.avi', recursive=True)
        avi_files = [file for file in avi_files if f"_{mouse_id}_" in os.path.basename(file)]
        processed_mice.add(mouse_id)  # Отмечаем идентификатор как обработанный
    
    if not avi_files:
        print(f"Видео не найдены для {mouse_id}. Пропуск.")
        continue

    # Сортировка файлов в алфавитном порядке
    avi_files = natsorted(avi_files)
    print(f"Найдено {len(avi_files)} видео для {mouse_id}:")
    print("\n".join(avi_files))

    # Объединение видео и сохранение
    clips = []
    for av in avi_files:
        clips.append(VideoFileClip(av))
    result = concatenate_videoclips(clips, method='compose')

    # Сохранение результата
    output_file = os.path.join(root, f"{exp}_{mouse_id}.mp4")
    result.write_videofile(output_file)
    print(f"Видео сохранено: {output_file}")
